# 🛰️ Spectrum-SLM — 3-Phase Training Pipeline

**Architecture (v2 — 176-patch tokenizer)**

| Component | Detail |
|---|---|
| Input | 176-bin PSD vector |
| Tokenizer | `PatchEmbedding(patch_size=1)` → **176 tokens** (1 bin each) + CLS → seq_len **177** |
| Positional Enc | Learnable + Sinusoidal (FrequencyAware), 177 positions |
| Encoder | 4-layer Transformer, 4 heads, d_model=128, FFN=512 |
| Heads | PU Detection · Modulation (4-class) · SNR Regression · Generative · MSM |

**Instructions:** Enable **GPU (T4×2)** + **Internet** in Kaggle settings, then click **Run All**.

In [ ]:
# 0. Environment check
import subprocess, sys
print('Python:', sys.version)
gpu_info = subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print('GPU:', gpu_info if gpu_info else 'No GPU detected — enable GPU in Kaggle settings!')
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# 1. Clone repository
!git clone https://github.com/31ASHISH/Spectrum-SLM.git /kaggle/working/Spectrum-SLM
%cd /kaggle/working/Spectrum-SLM/SDR_Data

In [ ]:
# 2. Install dependencies
!pip install -q streamlit scikit-learn
!npm install -g localtunnel

In [ ]:
# 3. Verify model architecture — 176 patches (patch_size=1, seq_len=177)
import sys
sys.path.insert(0, '/kaggle/working/Spectrum-SLM/SDR_Data')
import torch
from spectrum_slm_model import SpectrumSLM

model = SpectrumSLM()  # patch_size=1 by default
print(f'Total parameters : {model.count_parameters():,}')
print(f'n_patches        : {model.tokenizer.n_patches}   (expect 176)')
print(f'patch_size       : {model.tokenizer.patch_size}    (expect 1)')
print(f'Sequence length  : {model.tokenizer.n_patches + 1}  (expect 177 incl. CLS)')

# Quick forward-pass sanity check
B = 2
psd  = torch.randn(B, 176)
mask = torch.zeros(B, 176, dtype=torch.bool)
mask[:, ::3] = True  # mask every 3rd bin-patch
out  = model(psd, mask=mask, return_msm=True)
for k, v in out.items():
    print(f'  {k:12s}: {tuple(v.shape)}')
print('\n✅ Architecture verified — ready to train!')

In [ ]:
# 4. Connect Kaggle Input dataset to Config and run 3-phase training
# Make sure you clicked "Add Data" and selected your uploaded Dataset.
import os
dataset_path = "/kaggle/input"

if os.path.exists(dataset_path) and os.listdir(dataset_path):
    with open('config.py', 'a') as f:
        f.write(f'\n\nkaggle_override("{dataset_path}")\n')
    print("Starting end-to-end training pipeline...")
    !python training/run_3_phases.py
else:
    print(f"⚠️  Warning: No dataset found at {dataset_path}.")
    print("   → Go to Kaggle sidebar > Add Data, select your uploaded dataset, then re-run.")

In [ ]:
# 5. Post-training summary — list saved checkpoints
import os, glob
ckpt_dirs = ['checkpoints', 'slm_checkpoints', 'training']
found = []
for d in ckpt_dirs:
    found += glob.glob(f'{d}/**/*.pth', recursive=True)
    found += glob.glob(f'{d}/**/*.pt',  recursive=True)
if found:
    print(f'✅ {len(found)} checkpoint(s) found:')
    for f in found:
        size_mb = os.path.getsize(f) / 1e6
        print(f'  {f}  ({size_mb:.2f} MB)')
else:
    print('No checkpoints found yet — training may still be running.')

In [ ]:
# 6. Launch Streamlit app via localtunnel
import subprocess, time, re
st_proc = subprocess.Popen(
    ['streamlit', 'run', 'app_phase2.py', '--server.port', '8501', '--server.headless', 'true']
)
time.sleep(5)

print("Starting localtunnel...")
lt = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE, text=True)

for _ in range(15):
    line = lt.stdout.readline()
    m = re.search(r'https?://[\w\-.]+\.loca\.lt', line)
    if m:
        print(f'\n🚀 STREAMLIT LIVE AT: {m.group(0)}')
        print(f'🔑 Password required: {subprocess.getoutput("curl -s ifconfig.me")}\n')
        break
    time.sleep(1)